# Equipetrol Delivery · EC2 en Google Colab: la simulación completa, con GPU y sin GPU

Compila `delivery_sim` **dos veces en esta misma máquina**: con CUDA (`-DUSE_CUDA=ON`: R1 prefiltra las motos más
cercanas en la GPU en cada ronda del despachador y T1 calcula la alcanzabilidad al arrancar) y sin CUDA (todo en la
CPU, el build por defecto). Corre la simulación de la ciudad (4775 nodos, 3000 motos) y la de Equipetrol, 3 veces
cada una, **intercalando** las dos versiones, y mide con `tools/efecto_simulacion.py`: latencia de despacho por
pedido, pedidos asignados por segundo real, entregados por hora simulada, arranque y frames por segundo del bucle de
frames (`--frames`). Las aplicaciones de tipo A (el reporte) no corren durante la simulación: no entran acá.
Tarda unos 25 minutos.

**Antes de correr nada:** *Runtime → Change runtime type → T4 GPU → Save*; después *Runtime → Run all*.

**Al terminar:** descargar `results_simulacion.zip` (lo baja la última celda) y el notebook ejecutado (*File →
Download → Download .ipynb*). **No usar "Guardar en GitHub".**

In [1]:
# La máquina
!nvidia-smi --query-gpu=name,driver_version,clocks.max.sm,memory.total --format=csv
!nvcc --version | tail -2
!g++ --version | head -1
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Core'

name, driver_version, clocks.max.sm [MHz], memory.total [MiB]
Tesla T4, 580.82.07, 1590 MHz, 15360 MiB
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
CPU(s):                                  2
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
Thread(s) per core:                      2
Core(s) per socket:                      1


In [2]:
# El código: la rama de EC2 (el hash de src/ identifica lo medido)
BRANCH = "feat/ec2-gpu"
!rm -rf Delivery_Equipetrol && git clone -q -b {BRANCH} https://github.com/OjedaErwin123/Delivery_Equipetrol
!cd Delivery_Equipetrol && git log -1 --format='commit %h  %s' && echo "src/ = $(git rev-parse HEAD:src)"

commit 116dae1  wip: informe EC2 y efecto de la GPU en la simulación completa
src/ = 2e712d209fac355c261a7b3987d04150aec80234


In [3]:
# Dos builds, mismas banderas (Release, sin ventana): con CUDA y sin CUDA. Solo delivery_sim y dos pruebas.
!cmake -S Delivery_Equipetrol -B build-gpu -DUSE_CUDA=ON -DDELIVERY_WINDOW=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -1
!cmake -S Delivery_Equipetrol -B build-cpu -DDELIVERY_WINDOW=OFF -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -1
!cmake --build build-gpu -j2 --target delivery_sim frame_meter_check cli_check 2>&1 | grep -ciE 'warning' ; echo "^ warnings (gpu)"
!cmake --build build-cpu -j2 --target delivery_sim frame_meter_check cli_check 2>&1 | grep -ciE 'warning' ; echo "^ warnings (cpu)"
!./build-gpu/frame_meter_check | tail -1 && ./build-gpu/cli_check Delivery_Equipetrol/config/equipetrol.json | tail -1
!ldd build-cpu/delivery_sim | grep -c cuda ; echo "^ bibliotecas de CUDA en el build sin CUDA (debe ser 0)"

-- Build files have been written to: /content/build-gpu
-- Build files have been written to: /content/build-cpu
0
^ warnings (gpu)
0
^ warnings (cpu)
frames: 5, por segundo 19.51, intervalo mediana 35.00 ms, p99 100.00 ms, máximo 100.00 ms, de más de 50 ms: 1
si todos los nodos son de restaurantes, cualquier nodo sirve de destino: OK
0
^ bibliotecas de CUDA en el build sin CUDA (debe ser 0)


## Las corridas, intercaladas

Cada corrida: `delivery_sim <config> --headless --frames` (60 s reales). Sin ventana, `--frames` arma igual la foto de
cada frame (cada moto con su lock, lo que leería la ventana) sin dibujarla. Se intercalan GPU y CPU para que una
variación de la máquina (Colab es compartida) no caiga toda sobre una versión.

In [4]:
!rm -rf results && mkdir -p results/ciudad results/equipetrol
for r in range(3):
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-gpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json gpu 1 results/ciudad {r}
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-cpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json cpu 1 results/ciudad {r}
for r in range(3):
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-gpu/delivery_sim Delivery_Equipetrol/config/equipetrol.json gpu 1 results/equipetrol {r}
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py run build-cpu/delivery_sim Delivery_Equipetrol/config/equipetrol.json cpu 1 results/equipetrol {r}

gpu 0: 65.57 s reales, 1628 asignados, 1083 entregados, despacho p50 521.2 s simulados, 29.4 frames/s, consistente: yes
cpu 0: 67.85 s reales, 1417 asignados, 1186 entregados, despacho p50 559.8 s simulados, 28.9 frames/s, consistente: yes
gpu 1: 65.2 s reales, 1477 asignados, 1123 entregados, despacho p50 574.6 s simulados, 29.5 frames/s, consistente: yes
cpu 1: 60.84 s reales, 1290 asignados, 1166 entregados, despacho p50 523.0 s simulados, 29.2 frames/s, consistente: yes
gpu 2: 65.27 s reales, 1415 asignados, 1225 entregados, despacho p50 507.6 s simulados, 29.1 frames/s, consistente: yes
cpu 2: 61.96 s reales, 1250 asignados, 1143 entregados, despacho p50 501.6 s simulados, 29.0 frames/s, consistente: yes
gpu 0: 60.43 s reales, 143 asignados, 127 entregados, despacho p50 135.3 s simulados, 30.2 frames/s, consistente: yes
cpu 0: 60.01 s reales, 143 asignados, 127 entregados, despacho p50 135.4 s simulados, 30.2 frames/s, consistente: yes
gpu 1: 60.41 s reales, 143 asignados, 127 ent

In [5]:
# Lo que dijo cada versión en stderr (la primera corrida): GPU, tabla de T1, frames
!for f in results/*/*_stderr.txt; do echo "== $f"; cat $f; done

== results/ciudad/cpu_stderr.txt
delivery_sim: sin ventana: --headless; la simulación sigue igual
delivery_sim: sin ventana (la foto, sin dibujar), frames: 1728, por segundo 28.85, intervalo mediana 33.99 ms, p99 44.97 ms, máximo 55.99 ms, de más de 50 ms: 3
== results/ciudad/gpu_stderr.txt
delivery_sim: GPU: Tesla T4
delivery_sim: tabla de todos contra todos (4775 nodos) en la GPU: 353.326 ms de kernels, 437.449 ms en total
delivery_sim: sin ventana: --headless; la simulación sigue igual
delivery_sim: sin ventana (la foto, sin dibujar), frames: 1761, por segundo 29.40, intervalo mediana 33.63 ms, p99 38.95 ms, máximo 46.49 ms, de más de 50 ms: 0
== results/equipetrol/cpu_stderr.txt
delivery_sim: sin ventana: --headless; la simulación sigue igual
delivery_sim: sin ventana (la foto, sin dibujar), frames: 1812, por segundo 30.19, intervalo mediana 33.10 ms, p99 33.22 ms, máximo 34.38 ms, de más de 50 ms: 0
== results/equipetrol/gpu_stderr.txt
delivery_sim: GPU: Tesla T4
delivery_sim: tab

## nsys: el hilo dueño de la GPU durante la simulación

Una corrida con GPU de la ciudad bajo `nsys`: cuántas veces se lanzó cada kernel durante la simulación (R1 en cada
ronda del despachador, T1 una vez al arrancar), cuánto tardan y cuánto se copia. Bajo `nsys` la simulación corre más
lenta: estos números no van a la tabla, solo muestran qué hace la GPU.

In [6]:
!nsys profile --stats=true --force-overwrite true -o results/simulacion ./build-gpu/delivery_sim Delivery_Equipetrol/config/santa_cruz.json --headless --frames --log results/nsys.log 2>&1 | grep -A14 -E 'cuda_gpu_kern_sum|cuda_gpu_mem_time_sum|cuda_api_sum' | head -60
!rm -f results/nsys.log

[5/8] Executing 'cuda_api_sum' stats report

 Time (%)  Total Time (ns)  Num Calls    Avg (ns)    Med (ns)   Min (ns)   Max (ns)    StdDev (ns)                Name             
 --------  ---------------  ---------  ------------  ---------  --------  -----------  ------------  ------------------------------
     54.1      513,559,755      1,373     374,042.1  121,250.0     6,193  351,899,294   9,493,772.5  cudaEventSynchronize          
     21.6      205,129,817          8  25,641,227.1    5,694.5     3,528  204,771,765  72,379,754.0  cudaMalloc                    
     16.0      151,879,495      4,118      36,881.9   19,888.0    11,705   25,313,350     544,125.8  cudaMemcpy                    
      4.8       45,201,059      4,566       9,899.5    7,611.0     3,264      709,608      14,224.7  cudaLaunchKernel              
      1.5       14,408,746      2,746       5,247.2    4,463.0     2,584      940,814      18,044.7  cudaEventRecord               
      1.2       11,820,643     

## Tablas (desde los CSV)

Mediana [mín–máx] de las corridas. La de la laptop (sin GPU) está en el repo, en `docs/ec2/simulacion/laptop_*.csv`.

In [7]:
print("Ciudad (4775 nodos, 3000 motos)")
!python3 Delivery_Equipetrol/tools/efecto_simulacion.py table results/ciudad/simulacion.csv
print("Equipetrol (154 nodos, 8 motos)")
!python3 Delivery_Equipetrol/tools/efecto_simulacion.py table results/equipetrol/simulacion.csv
import glob
for f in sorted(glob.glob("Delivery_Equipetrol/docs/ec2/simulacion/laptop_*.csv")):
    print("Laptop:", f)
    !python3 Delivery_Equipetrol/tools/efecto_simulacion.py table {f}

Ciudad (4775 nodos, 3000 motos)
| máquina | versión | corridas | arranque (ms reales) | asignados / s real | despacho p50 (s sim.) | despacho p90 (s sim.) | entregados / h sim. | frames / s | frame p99 (ms) | frame máx. (ms) | fallbacks | ¿consistente? |
|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|:---:|
| Intel(R) Xeon(R) CPU @ 2.00GHz x2 | Tesla T4 | gpu | 3 | 858.8 [851.8–889.1] | 22.6 [21.7–24.8] | 521.2 [507.6–574.6] | 597.2 [595.3–599.5] | 1123.0 [1083.0–1225.0] | 29.4 [29.1–29.5] | 39.0 [38.0–39.9] | 43.1 [43.1–46.5] | 0 | sí |
| Intel(R) Xeon(R) CPU @ 2.00GHz x2 | Tesla T4 | cpu | 3 | 149.9 [147.5–157.6] | 20.9 [20.2–21.2] | 523.0 [501.6–559.8] | 598.2 [595.3–598.7] | 1166.0 [1143.0–1186.0] | 29.0 [28.9–29.2] | 40.9 [39.2–45.0] | 46.6 [46.0–56.0] | 0 | sí |
Equipetrol (154 nodos, 8 motos)
| máquina | versión | corridas | arranque (ms reales) | asignados / s real | despacho p50 (s sim.) | despacho p90 (s sim.) | entregados / h sim. | frames / s | frame p99 (ms) | 

In [8]:
GUARDAR_EN_DRIVE = False
!cd Delivery_Equipetrol && git rev-parse HEAD:src > ../results/src_tree.txt
!zip -qr results_simulacion.zip results && ls -la results_simulacion.zip
if GUARDAR_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/EC2_resultados && cp results_simulacion.zip /content/drive/MyDrive/EC2_resultados/
from google.colab import files
files.download("results_simulacion.zip")

-rw-r--r-- 1 root root 12935097 Nov 21 02:33 results_simulacion.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>